<a href="https://colab.research.google.com/github/abyanrizz/practicalstatisticfordatascientist/blob/main/Chapter_5_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 5
Data scientists are often tasked with automating decisions for business problems. Is
an email an attempt at phishing? Is a customer likely to churn? Is the web user likely
to click on an advertisement? These are all classification problems, a form of super‐
vised learning in which we first train a model on data where the outcome is known
and then apply the model to data where the outcome is not known. Classification is
perhaps the most important form of prediction: the goal is to predict whether a
record is a 1 or a 0 (phishing/not-phishing, click/don’t click, churn/don’t churn), or in
some cases, one of several categories (for example, Gmail’s filtering of your inbox into
“primary,” “social,” “promotional,” or “forums”).
Often, we need more than a simple binary classification: we want to know the predic‐
ted probability that a case belongs to a class. Rather than having a model simply
assign a binary classification, most algorithms can return a probability score (propen‐
sity) of belonging to the class of interest. In fact, with logistic regression, the default
output from R is on the log-odds scale, and this must be transformed to a propensity.
In Python’s scikit-learn, logistic regression, like most classification methods, pro‐
vides two prediction methods: predict (which returns the class) and predict_proba
(which returns probabilities for each class). A sliding cutoff can then be used to con‐
vert the propensity score to a decision. The general approach is as follows:
1. Establish a cutoff probability for the class of interest, above which we consider a
record as belonging to that class.
2. Estimate (with any model) the probability that a record belongs to the class of
interest.
3. If that probability is above the cutoff probability, assign the new record to the
class of interest.

195

1 This and subsequent sections in this chapter © 2020 Datastats, LLC, Peter Bruce, Andrew Bruce, and Peter
Gedeck; used with permission.
The higher the cutoff, the fewer the records predicted as 1—that is, as belonging to
the class of interest. The lower the cutoff, the more the records predicted as 1.
This chapter covers several key techniques for classification and estimating propensi‐
ties; additional methods that can be used both for classification and for numerical
prediction are described in the next chapter.

More Than Two Categories?

The vast majority of problems involve a binary response. Some classification prob‐
lems, however, involve a response with more than two possible outcomes. For exam‐
ple, at the anniversary of a customer’s subscription contract, there might be three
outcomes: the customer leaves or “churns” (Y = 2), goes on a month-to-month con‐
tract (Y = 1), or signs a new long-term contract (Y = 0). The goal is to predict Y = j for
j = 0, 1, or 2. Most of the classification methods in this chapter can be applied, either
directly or with modest adaptations, to responses that have more than two outcomes.
Even in the case of more than two outcomes, the problem can often be recast into a
series of binary problems using conditional probabilities. For example, to predict the
outcome of the contract, you can solve two binary prediction problems:
• Predict whether Y = 0 or Y > 0.
• Given that Y > 0, predict whether Y = 1 or Y = 2.
In this case, it makes sense to break up the problem into two cases: (1) whether the
customer churns; and (2) if they don’t churn, what type of contract they will choose.
From a model-fitting viewpoint, it is often advantageous to convert the multiclass
problem to a series of binary problems. This is particularly true when one category is
much more common than the other categories.
Naive Bayes
The naive Bayes algorithm uses the probability of observing predictor values, given
an outcome, to estimate what is really of interest: the probability of observing out‐
come Y = i, given a set of predictor values.1

196 | Chapter 5: Classification

Key Terms for Naive Bayes

Conditional probability
The probability of observing some event (say, X = i) given some other event (say,
Y = i), written as P Xi
Yi
.
Posterior probability
The probability of an outcome after the predictor information has been incorpo‐
rated (in contrast to the prior probability of outcomes, not taking predictor infor‐
mation into account).
To understand naive Bayesian classification, we can start out by imagining complete
or exact Bayesian classification. For each record to be classified:
1. Find all the other records with the same predictor profile (i.e., where the predic‐
tor values are the same).
2. Determine what classes those records belong to and which class is most prevalent
(i.e., probable).
3. Assign that class to the new record.
The preceding approach amounts to finding all the records in the sample that are
exactly like the new record to be classified in the sense that all the predictor values are
identical.

Predictor variables must be categorical (factor) variables in the
standard naive Bayes algorithm. See “Numeric Predictor Variables”
on page 200 for two workarounds for using continuous variables.

Why Exact Bayesian Classification Is Impractical
When the number of predictor variables exceeds a handful, many of the records to be
classified will be without exact matches. Consider a model to predict voting on the
basis of demographic variables. Even a sizable sample may not contain even a single
match for a new record who is a male Hispanic with high income from the US Mid‐
west who voted in the last election, did not vote in the prior election, has three
daughters and one son, and is divorced. And this is with just eight variables, a small
number for most classification problems. The addition of just a single new variable
with five equally frequent categories reduces the probability of a match by a factor
of 5.

Naive Bayes | 197

The Naive Solution
In the naive Bayes solution, we no longer restrict the probability calculation to those
records that match the record to be classified. Instead, we use the entire data set. The
naive Bayes modification is as follows:
1. For a binary response Y = i (i = 0 or 1), estimate the individual conditional prob‐
abilities for each predictor P Xj

Y = i ; these are the probabilities that the pre‐
dictor value is in the record when we observe Y = i. This probability is estimated
by the proportion of Xj

values among the Y = i records in the training set.
2. Multiply these probabilities by each other, and then by the proportion of records
belonging to Y = i.
3. Repeat steps 1 and 2 for all the classes.
4. Estimate a probability for outcome i by taking the value calculated in step 2 for
class i and dividing it by the sum of such values for all classes.
5. Assign the record to the class with the highest probability for this set of predictor
values.
This naive Bayes algorithm can also be stated as an equation for the probability of
observing outcome Y = i, given a set of predictor values X1
, ⋯, Xp
:

P(Y = i|X1
, X2
, ..., Xp
)

Here is the full formula for calculating class probabilities using exact Bayes
classification:

P(Y = i|X1
, X2
, ..., Xp
) =

P(Y = i)P(X1
, ..., Xp
|Y = i)

P(Y = 0)P(X1
, ..., Xp
|Y = 0) + P(Y = 1)P(X1
, ..., Xp
|Y = 1)
Under the naive Bayes assumption of conditional independence, this equation
changes into:
P(Y = i|X1
, X2
, ..., Xp
)

=

P(Y = i)P(X1

|Y = i)...P(Xp
|Y = i)

P(Y = 0)P(X1

|Y = 0)...P(Xp

|Y = 0) + P(Y = 1)P(X1

|Y = 1)...P(Xp
|Y = 1)
Why is this formula called “naive”? We have made a simplifying assumption that the
exact conditional probability of a vector of predictor values, given observing an

198 | Chapter 5: Classification

outcome, is sufficiently well estimated by the product of the individual conditional
probabilities P Xj

Y = i . In other words, in estimating P Xj

Y = i instead of

P X1
, X2
, ⋯Xp
Y = i , we are assuming Xj

is independent of all the other predictor

variables Xk

for k ≠ j.

Several packages in R can be used to estimate a naive Bayes model. The following fits
a model to the loan payment data using the klaR package:

In [ ]:
library(klaR)
naive_model <- NaiveBayes(outcome ~ purpose_ + home_ + emp_len_,
data = na.omit(loan_data))
naive_model$table
$purpose_
var
grouping credit_card debt_consolidation home_improvement major_purchase
paid off 0.18759649 0.55215915 0.07150104 0.05359270
default 0.15151515 0.57571347 0.05981209 0.03727229
var
grouping medical other small_business
paid off 0.01424728 0.09990737 0.02099599
default 0.01433549 0.11561025 0.04574126
$home_
var
grouping MORTGAGE OWN RENT
paid off 0.4894800 0.0808963 0.4296237
default 0.4313440 0.0832782 0.4853778
$emp_len_
var
grouping < 1 Year > 1 Year
paid off 0.03105289 0.96894711
default 0.04728508 0.95271492

The output from the model is the conditional probabilities P Xj
Y = i .

In Python we can use sklearn.naive_bayes.MultinomialNB from scikit-learn. We
need to convert the categorical features to dummy variables before we fit the model:

In [ ]:
predictors = ['purpose_', 'home_', 'emp_len_']
outcome = 'outcome'
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='')
y = loan_data[outcome]
naive_model = MultinomialNB(alpha=0.01, fit_prior=True)
naive_model.fit(X, y)

It is possible to derive the conditional probabilities from the fitted model using the
property feature_log_prob_.

In [ ]:
new_loan <- loan_data[147, c('purpose_', 'home_', 'emp_len_')]
row.names(new_loan) <- NULL
new_loan
purpose_ home_ emp_len_
1 small_business MORTGAGE > 1 Year

The prediction also returns a posterior estimate of the probability of default. The
naive Bayesian classifier is known to produce biased estimates. However, where the
goal is to rank records according to the probability that Y = 1, unbiased estimates of
probability are not needed, and naive Bayes produces good results.
Numeric Predictor Variables
The Bayesian classifier works only with categorical predictors (e.g., with spam classi‐
fication, where the presence or absence of words, phrases, characters, and so on lies at
the heart of the predictive task). To apply naive Bayes to numerical predictors, one of
two approaches must be taken:

200 | Chapter 5: Classification

2 It is certainly surprising that the first article on statistical classification was published in a journal devoted to
eugenics. Indeed, there is a disconcerting connection between the early development of statistics and eugen‐
ics.
• Bin and convert the numerical predictors to categorical predictors and apply the
algorithm of the previous section.
• Use a probability model—for example, the normal distribution (see “Normal
Distribution” on page 69)—to estimate the conditional probability P Xj
Y = i .

When a predictor category is absent in the training data, the algo‐
rithm assigns zero probability to the outcome variable in new data,
rather than simply ignoring this variable and using the information
from other variables, as other methods might. Most implementa‐
tions of Naive Bayes use a smoothing parameter (Laplace Smooth‐
ing) to prevent this.

Key Ideas

• Naive Bayes works with categorical (factor) predictors and outcomes.
• It asks, “Within each outcome category, which predictor categories are most
probable?”
• That information is then inverted to estimate probabilities of outcome categories,
given predictor values.

Further Reading
• The Elements of Statistical Learning, 2nd ed., by Trevor Hastie, Robert Tibshirani,
and Jerome Friedman (Springer, 2009).
• There is a full chapter on naive Bayes in Data Mining for Business Analytics by
Galit Shmueli, Peter Bruce, Nitin Patel, Peter Gedeck, Inbal Yahav, and Kenneth
Lichtendahl (Wiley, 2007–2020, with editions for R, Python, Excel, and JMP).
Discriminant Analysis
Discriminant analysis is the earliest statistical classifier; it was introduced